<a href="https://colab.research.google.com/github/HarshaniDil/3D_modeling/blob/main/solarThermalHeatingAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection  import train_test_split
from sklearn.pipeline         import make_pipeline
from sklearn.preprocessing    import StandardScaler, PolynomialFeatures
from sklearn.decomposition    import PCA
from sklearn.linear_model     import LinearRegression
from sklearn.metrics          import r2_score, mean_squared_error
from scipy.optimize           import curve_fit

In [23]:
# Evaluation metrics helper
# ============================================================
def evaluate(y_true, y_pred, k, label=""):
    """
    Compute evaluation metrics.
    y_true : actual values
    y_pred : predicted values
    k      : number of model parameters (incl. intercept)
    """
    m    = len(y_true)
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rss  = np.sum((y_true - y_pred) ** 2)
    bic  = m * np.log(rss / m) + k * np.log(m)

    if label:
        print(f"  R²   = {r2:.4f}")
        print(f"  RMSE = {rmse:.2f} kWh/day")
        print(f"  BIC  = {bic:.1f}")
    return {"R2": r2, "RMSE": rmse, "BIC": bic}


In [36]:
# 1. Load data
# ============================================================
df = pd.read_csv('./Heating-data.csv', sep='\t')
df.columns = ['Date', 'Sun', 'Temp', 'Solar', 'Pump', 'Valve', 'Gas']

print("=" * 55)
print("DATA OVERVIEW")
print("=" * 55)
print(f"Total observations : {len(df)}")
print(f"Training data (70%): {int(len(df)*0.7)}")
print(f"Test data (30%)    : {int(len(df)*0.3)}")

DATA OVERVIEW
Total observations : 348
Training data (70%): 243
Test data (30%)    : 104


In [37]:
# 2. Scatter plots
# ============================================================
features_all = ['Sun', 'Temp', 'Solar', 'Pump', 'Valve']
labels = {
    'Sun':   'Sunshine duration [h/day]',
    'Temp':  'Outdoor temperature [°C]',
    'Solar': 'Solar yield [kWh/day]',
    'Pump':  'Solar pump [h/day]',
    'Valve': 'Valve [h/day]',
    'Gas':   'Gas consumption [kWh/day]'
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, features_all):
    ax.scatter(df[feat], df['Gas'], alpha=0.4, s=10, color='steelblue')
    ax.set_xlabel(labels[feat])
    ax.set_ylabel(labels['Gas'])
    ax.set_title(f'Gas vs {feat}')
plt.suptitle('Scatter Plots: Gas Consumption vs Each Predictor', fontsize=13)
plt.tight_layout()
plt.savefig('01_scatter_plots.png', dpi=150)
plt.close()
print("\nSaved: 01_scatter_plots.png")



Saved: 01_scatter_plots.png


In [38]:
# 3. Correlation matrix
# ============================================================
corr = df[features_all + ['Gas']].corr().round(2)
print("\nCorrelation matrix:")
print(corr)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}',
                ha='center', va='center', fontsize=9)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('02_correlation_matrix.png', dpi=150)
plt.close()
print("Saved: 02_correlation_matrix.png")


Correlation matrix:
        Sun  Temp  Solar  Pump  Valve   Gas
Sun    1.00  0.65   0.90  0.89   0.57 -0.65
Temp   0.65  1.00   0.77  0.76   0.60 -0.93
Solar  0.90  0.77   1.00  0.98   0.66 -0.79
Pump   0.89  0.76   0.98  1.00   0.67 -0.80
Valve  0.57  0.60   0.66  0.67   1.00 -0.67
Gas   -0.65 -0.93  -0.79 -0.80  -0.67  1.00
Saved: 02_correlation_matrix.png


In [39]:
# 4. PCA on solar group (Sun, Solar, Pump)
# ============================================================
solar_features = ['Sun', 'Solar', 'Pump']
X_solar        = np.c_[df[solar_features]]
scaler_solar   = StandardScaler()
X_solar_scaled = scaler_solar.fit_transform(X_solar)

In [40]:
# Full PCA for scree plot
pca_full = PCA(3)
pca_full.fit(X_solar_scaled)
explained = pca_full.explained_variance_ratio_

print("\nPCA on solar group:")
for i, ev in enumerate(explained):
    print(f"  PC{i+1}: {ev*100:.1f}%")



PCA on solar group:
  PC1: 95.0%
  PC2: 4.4%
  PC3: 0.6%


In [41]:
# Scree plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar([f'PC{i+1}' for i in range(3)], explained * 100,
       color='steelblue', edgecolor='black')
ax.set_ylabel('Explained variance [%]')
ax.set_title('Scree Plot – PCA on Solar Group\n(Sun, Solar yield, Solar pump)')
for i, ev in enumerate(explained):
    ax.text(i, ev*100 + 0.5, f'{ev*100:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('03_scree_plot.png', dpi=150)
plt.close()
print("Saved: 03_scree_plot.png")


Saved: 03_scree_plot.png


In [42]:
# Keep PC1 only
pca1 = PCA(1)
pca1.fit(X_solar_scaled)
print("\nPC1 loadings:")
for feat, load in zip(solar_features, pca1.components_[0]):
    print(f"  {feat}: {load:.3f}")


PC1 loadings:
  Sun: 0.565
  Solar: 0.584
  Pump: 0.583


In [43]:
# 5. Prepare predictors and train/test split
# ============================================================
PC1_all = pca1.transform(X_solar_scaled)
X_all   = np.column_stack([PC1_all[:,0],
                            df['Temp'].values,
                            df['Valve'].values])
y_all   = df['Gas'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.30, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


Train: 243 | Test: 105


In [44]:
# 6. Model A — Linear Regression (Pipeline)
# ============================================================
pipe_A = make_pipeline(StandardScaler(), LinearRegression())
pipe_A.fit(X_train, y_train)

y_pred_A_train = pipe_A.predict(X_train)
y_pred_A_test  = pipe_A.predict(X_test)

# Get the LinearRegression step from pipeline
lr = pipe_A.named_steps['linearregression']

print("Intercept:", lr.intercept_)
print("Coefficients:", lr.coef_)

print("\n" + "=" * 55)
print("MODEL A – Linear Regression (Pipeline)")
print("  Predictors: PC1_solar, Temp, Valve")
print("=" * 55)
print("  --- Train ---")
m_A_train = evaluate(y_train, y_pred_A_train, X_all.shape[1]+1, label="train")
print("  --- Test ---")
m_A_test  = evaluate(y_test,  y_pred_A_test,  X_all.shape[1]+1, label="test")


Intercept: 223.1670781893004
Coefficients: [ -16.81707286 -114.32272521  -23.09586766]

MODEL A – Linear Regression (Pipeline)
  Predictors: PC1_solar, Temp, Valve
  --- Train ---
  R²   = 0.8931
  RMSE = 49.48 kWh/day
  BIC  = 1918.1
  --- Test ---
  R²   = 0.8790
  RMSE = 51.80 kWh/day
  BIC  = 847.6


In [45]:
# Residual plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y_pred_A_test, y_test - y_pred_A_test,
           alpha=0.4, s=10, color='steelblue')
ax.axhline(0, color='red', linewidth=1)
ax.set_xlabel('Predicted Gas consumption [kWh/day]')
ax.set_ylabel('Residuals [kWh/day]')
ax.set_title('Residuals – Model A: Linear Regression (Test Data)')
plt.tight_layout()
plt.savefig('04_residuals_linear.png', dpi=150)
plt.close()
print("Saved: 04_residuals_linear.png")


Saved: 04_residuals_linear.png


In [46]:
# 7. Model B — Polynomial Regression deg.2 (Pipeline)
# ============================================================
pipe_B = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LinearRegression()
)
pipe_B.fit(X_train, y_train)

y_pred_B_train = pipe_B.predict(X_train)
y_pred_B_test  = pipe_B.predict(X_test)
k_B = pipe_B.named_steps['linearregression'].coef_.shape[0] + 1

print("\n" + "=" * 55)
print("MODEL B – Polynomial deg.2 (Pipeline)")
print("  PolynomialFeatures on PC1_solar, Temp, Valve")
print("=" * 55)
print(f"  Parameters k = {k_B}")
print("  --- Train ---")
m_B_train = evaluate(y_train, y_pred_B_train, k_B, label="train")
print("  --- Test ---")
m_B_test  = evaluate(y_test,  y_pred_B_test,  k_B, label="test")



MODEL B – Polynomial deg.2 (Pipeline)
  PolynomialFeatures on PC1_solar, Temp, Valve
  Parameters k = 10
  --- Train ---
  R²   = 0.9291
  RMSE = 40.28 kWh/day
  BIC  = 1851.1
  --- Test ---
  R²   = 0.9080
  RMSE = 45.18 kWh/day
  BIC  = 846.8


In [47]:
# 8. Model C — Theory-Based HDD (curve_fit)
# ============================================================
x_temp_train = X_train[:, 1]
x_temp_test  = X_test[:,  1]
x_temp_all   = X_all[:,   1]

def f_hdd(T, a, b, T_base):
    """Heating Degree-Day model: Gas = a + b*max(0, T_base - T)"""
    return a + b * np.maximum(0, T_base - T)

coefs, _ = curve_fit(f_hdd, x_temp_train, y_train, p0=[100, 20, 15])
y_pred_C_train = f_hdd(x_temp_train, *coefs)
y_pred_C_test  = f_hdd(x_temp_test,  *coefs)
y_pred_C_all   = f_hdd(x_temp_all,   *coefs)

print("\n" + "=" * 55)
print("MODEL C – HDD Nonlinear (curve_fit)")
print("  Gas = a + b·max(0, T_base − Temp)")
print("=" * 55)
print(f"  a      = {coefs[0]:.2f} kWh/day")
print(f"  b      = {coefs[1]:.2f} kWh/day/°C")
print(f"  T_base = {coefs[2]:.2f} °C")
print("  --- Train ---")
m_C_train = evaluate(y_train, y_pred_C_train, 3, label="train")
print("  --- Test ---")
m_C_test  = evaluate(y_test,  y_pred_C_test,  3, label="test")



MODEL C – HDD Nonlinear (curve_fit)
  Gas = a + b·max(0, T_base − Temp)
  a      = 69.05 kWh/day
  b      = 28.55 kWh/day/°C
  T_base = 16.16 °C
  --- Train ---
  R²   = 0.9037
  RMSE = 46.96 kWh/day
  BIC  = 1887.2
  --- Test ---
  R²   = 0.8856
  RMSE = 50.37 kWh/day
  BIC  = 837.0


In [48]:
# HDD fit plot
T_sorted = np.linspace(x_temp_all.min(), x_temp_all.max(), 300)
y_curve  = f_hdd(T_sorted, *coefs)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_temp_all, y_all, alpha=0.4, s=10,
           color='steelblue', label='Observed data')
ax.plot(T_sorted, y_curve, color='red', linewidth=2,
        label=f'HDD: a={coefs[0]:.0f}, b={coefs[1]:.1f}, T_base={coefs[2]:.1f}°C')
ax.axvline(coefs[2], color='gray', linestyle='--', linewidth=1,
           label=f'T_base = {coefs[2]:.1f}°C')
ax.set_xlabel('Outdoor temperature [°C]')
ax.set_ylabel('Gas consumption [kWh/day]')
ax.set_title(f'Model C – Heating Degree-Day Model\nR² test = {m_C_test["R2"]:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('06_hdd_fit.png', dpi=150)
plt.close()
print("Saved: 06_hdd_fit.png")


Saved: 06_hdd_fit.png


In [49]:
# 9. Model comparison — R², RMSE, BIC
# ============================================================
models = [
    'Model A\nLinear\n(PC1+Temp+Valve)',
    'Model B\nPolynomial\ndeg.2',
    'Model C\nHDD\ncurve_fit'
]
colors = ['steelblue', 'darkorange', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R² Test
r2_test_vals = [m_A_test['R2'], m_B_test['R2'], m_C_test['R2']]
axes[0].bar(models, r2_test_vals, color=colors, edgecolor='black')
axes[0].set_ylabel('R² Test')
axes[0].set_ylim(0.85, 0.95)
axes[0].set_title('R² (Test Data)\nhigher = better')
for i, v in enumerate(r2_test_vals):
    axes[0].text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=10)

# RMSE Test
rmse_test_vals = [m_A_test['RMSE'], m_B_test['RMSE'], m_C_test['RMSE']]
axes[1].bar(models, rmse_test_vals, color=colors, edgecolor='black')
axes[1].set_ylabel('RMSE [kWh/day]')
axes[1].set_title('RMSE (Test Data)\nlower = better')
for i, v in enumerate(rmse_test_vals):
    axes[1].text(i, v + 0.3, f'{v:.2f}', ha='center', fontsize=10)

# BIC
bic_vals = [m_A_test['BIC'], m_B_test['BIC'], m_C_test['BIC']]
axes[2].bar(models, bic_vals, color=colors, edgecolor='black')
axes[2].set_ylabel('BIC')
axes[2].set_title('BIC\nlower = better')
for i, v in enumerate(bic_vals):
    axes[2].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=10)

plt.suptitle('Model Comparison – R² Test, RMSE Test, BIC', fontsize=13)
plt.tight_layout()
plt.savefig('05_model_comparison.png', dpi=150)
plt.close()
print("Saved: 05_model_comparison.png")


Saved: 05_model_comparison.png


In [50]:
# 10. Final summary
# ============================================================
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"{'Metric':<20} {'Model A':>12} {'Model B':>12} {'Model C':>12}")
print("-" * 65)
print(f"{'R² (train)':<20} {m_A_train['R2']:>12.3f} {m_B_train['R2']:>12.3f} {m_C_train['R2']:>12.3f}")
print(f"{'RMSE (train)':<20} {m_A_train['RMSE']:>12.2f} {m_B_train['RMSE']:>12.2f} {m_C_train['RMSE']:>12.2f}")
print(f"{'BIC':<20} {m_A_train['BIC']:>12.1f} {m_B_train['BIC']:>12.1f} {m_C_train['BIC']:>12.1f}")
print("-" * 65)
print(f"{'R² (test)':<20} {m_A_test['R2']:>12.3f} {m_B_test['R2']:>12.3f} {m_C_test['R2']:>12.3f}")
print(f"{'RMSE (test)':<20} {m_A_test['RMSE']:>12.2f} {m_B_test['RMSE']:>12.2f} {m_C_test['RMSE']:>12.2f}")
print(f"{'BIC':<20} {m_A_test['BIC']:>12.1f} {m_B_test['BIC']:>12.1f} {m_C_test['BIC']:>12.1f}")
print("=" * 65)
print("→ Best model: B — highest R² test, lowest RMSE test, lowest BIC")



FINAL SUMMARY
Metric                    Model A      Model B      Model C
-----------------------------------------------------------------
R² (train)                  0.893        0.929        0.904
RMSE (train)                49.48        40.28        46.96
BIC                        1918.1       1851.1       1887.2
-----------------------------------------------------------------
R² (test)                   0.879        0.908        0.886
RMSE (test)                 51.80        45.18        50.37
BIC                         847.6        846.8        837.0
→ Best model: B — highest R² test, lowest RMSE test, lowest BIC
